In [1]:
import pandas as pd
file = "yellow_tripdata_2026-01.parquet"
df = pd.read_parquet(file)
print(df.head(5))

   VendorID tpep_pickup_datetime tpep_dropoff_datetime  passenger_count  \
0         2  2026-01-01 00:54:04   2026-01-01 00:59:37              1.0   
1         1  2026-01-01 00:34:04   2026-01-01 00:39:47              0.0   
2         1  2026-01-01 00:57:06   2026-01-01 01:05:59              0.0   
3         2  2026-01-01 00:15:22   2026-01-01 00:58:10              4.0   
4         2  2026-01-01 00:27:13   2026-01-01 00:40:43              0.0   

   trip_distance  RatecodeID store_and_fwd_flag  PULocationID  DOLocationID  \
0           0.97         1.0                  N           239           238   
1           0.90         1.0                  N           163           162   
2           1.40         1.0                  N            43           237   
3           5.58         1.0                  N           142           209   
4           2.16         1.0                  N            88           144   

   payment_type  fare_amount  extra  mta_tax  tip_amount  tolls_amount  \


# Pyspark vs Pandas

In [ ]:

import time
import psutil
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg

file = "yellow_tripdata_2026-01.parquet"

process = psutil.Process(os.getpid())

# ======================
# 🐼 PANDAS
# ======================
mem_before = process.memory_info().rss / 1024 / 1024
start = time.time()

df_pandas = pd.read_parquet(file)
avg_fare_pandas = df_pandas["fare_amount"].mean()

end = time.time()
mem_after = process.memory_info().rss / 1024 / 1024

print("PANDAS")
print("Avg fare:", avg_fare_pandas)
print("Temps:", round(end - start, 2), "s")
print("RAM:", round(mem_after - mem_before, 2), "MB")

print("##########################################################")

# ======================
# ⚡ PYSPARK
# ======================

# démarrage Spark (hors mesure)
spark = SparkSession.builder \
    .appName("compare") \
    .master("local[*]") \
    .getOrCreate()

start = time.time()

df_spark = spark.read.parquet(file)
avg_fare_spark = df_spark.select(avg("fare_amount")).collect()[0][0]

end = time.time()

print("PYSPARK")
print("Avg fare:", avg_fare_spark)
print("Temps:", round(end - start, 2), "s")
print("RAM: non fiable (Spark utilise JVM)")

spark.stop()

PANDAS
Avg fare: 20.804253893203256
Temps: 0.26 s
RAM: 603.03 MB
##########################################################


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 09:57:22 WARN Utils: Your hostname, PYTHON-08, resolves to a loopback address: 127.0.1.1; using 172.22.114.75 instead (on interface eno1)
26/04/17 09:57:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 09:57:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


PYSPARK
Avg fare: 20.804253893199448
Temps: 3.17 s
RAM: non fiable (Spark utilise JVM)


# RDD et lazy evaluation 

In [ ]:
from pyspark import SparkContext


#sc = SparkContext("local", "Lazy Test")
sc = SparkContext("local", "RDD example")

l = [1, 2, 3, 4, 5]

rdd = sc.parallelize(l)

rdd2 = rdd.map(lambda x: x * 2)

#print(rdd)
print(rdd.collect())
time.sleep(20)
sc.stop()



NameError: name 'sc' is not defined

### 12. Explorer glom() (distribution par partition)

In [ ]:

#sc = SparkContext("local[2]", "glom_demo")

rdd = sc.parallelize([1, 2, 3, 4, 5, 6, 7, 8], 3)
print(rdd.glom().collect())
rdd2 = rdd.map(lambda x: x * 10)

print(rdd2.glom().collect())
time.sleep(20)
sc.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/17 09:58:33 WARN Utils: Your hostname, PYTHON-08, resolves to a loopback address: 127.0.1.1; using 172.22.114.75 instead (on interface eno1)
26/04/17 09:58:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 09:58:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


[[1, 2], [3, 4], [5, 6, 7, 8]]
[[10, 20], [30, 40], [50, 60, 70, 80]]


NameError: name 'time' is not defined

### 13. Passer à [1,2,3,4,5]*1_000_000 – tester map, filter, reduce

In [ ]:

sc = SparkContext("local[*]", "big_rdd_test")

data = [1, 2, 3, 4, 5] * 1_000_000

rdd = sc.parallelize(data)
rdd_map = rdd.map(lambda x: x * 2)
rdd_filter = rdd_map.filter(lambda x: x > 5)
result = rdd_filter.reduce(lambda a, b: a + b)

print(result)

ValueError: Cannot run multiple SparkContexts at once; existing SparkContext(app=glom_demo, master=local[2]) created by __init__ at /tmp/ipykernel_50608/836671174.py:1 

### Challenge – Arbres remarquables de Paris
### 14. Télécharger le dataset (lien ci-dessous) et répondre en utilisant uniquement l'API RDD :
### ◦ Quel est l'âge du plus vieil arbre ?
### ◦ Quelle est la hauteur moyenne de tous les arbres ?
### ◦ Dans quel arrondissement y a-t-il le plus d'arbres ?

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ArbresRemarquables").getOrCreate()
#sc = spark.sparkContext

# Charger le parquet via DataFrame puis convertir en RDD
df = spark.read.parquet("arbresremarquablesparis.parquet")
rdd = df.rdd

# Afficher les colonnes disponibles
print(df.columns)

In [ ]:
from pyspark.sql import SparkSession
from datetime import datetime

spark = SparkSession.builder.appName("ArbresRemarquables").getOrCreate()

df = spark.read.parquet("arbresremarquablesparis.parquet")
rdd = df.rdd

annee_actuelle = 2026

# ── 1. Âge du plus vieil arbre ──────────────────────────────────────────────
def annee_valide(row):
    val = row["com_annee_plantation"]
    if val is None:
        return False
    try:
        int(val)
        return True
    except (ValueError, TypeError):
        return False

age_max = (
    rdd
    .filter(annee_valide)
    .map(lambda row: annee_actuelle - int(row["com_annee_plantation"]))
    .reduce(lambda a, b: a if a > b else b)
)
print(f"Âge du plus vieil arbre : {age_max} ans")


# ── 2. Hauteur moyenne de tous les arbres ───────────────────────────────────
def hauteur_valide(row):
    val = row["arbres_hauteurenm"]
    if val is None:
        return False
    try:
        float(val)
        return True
    except (ValueError, TypeError):
        return False

rdd_hauteur = (
    rdd
    .filter(hauteur_valide)
    .map(lambda row: (float(row["arbres_hauteurenm"]), 1))
)

total_hauteur, total_count = rdd_hauteur.reduce(lambda a, b: (a[0]+b[0], a[1]+b[1]))
moyenne = total_hauteur / total_count
print(f"Hauteur moyenne : {moyenne:.2f} m")


# ── 3. Arrondissement avec le plus d'arbres ─────────────────────────────────
def arrond_valide(row):
    val = row["arbres_arrondissement"]
    if val is None:
        return False
    try:
        str(val).strip()
        return True
    except (ValueError, TypeError):
        return False

arrond_max = (
    rdd
    .filter(arrond_valide)
    .map(lambda row: (row["arbres_arrondissement"], 1))
    .reduceByKey(lambda a, b: a + b)
    .reduce(lambda a, b: a if a[1] > b[1] else b)
)
print(f"Arrondissement avec le plus d'arbres : {arrond_max[0]} ({arrond_max[1]} arbres)")